# Scenario C: Assistive Household Robot
## Full-Stack Mobile Manipulator (CS4973)

A simulated mobile manipulator written in pure Python. The mobile base drives across a
household room, works out where it is, and builds a map as it goes. Once it arrives, the arm
does a contact task (pressing a light switch, or handing an item to a person) using admittance
control so that it gives way if it touches something unexpectedly.

**Contents**

| Task | Section |
|---|---|
| 1 | Kinematic motion model (unicycle plus a differential drive layer) |
| 2 | Environment and sensor simulation (room, odometry noise, landmarks, lidar) |

The notebook runs top to bottom. Each task ends with its own block of tests, and there is a
combined run of every test at the end, so a clean execution of the notebook is also a passing
test run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from dataclasses import dataclass, field, replace
from typing import List, Tuple, Optional

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 9,
})

# Every random draw in the project comes from a stream derived from this one number, so the
# whole notebook is reproducible from a single seed. Change it to get a different run.
MASTER_SEED = 7

def make_rng(stream: int) -> np.random.Generator:
    """Return an independent random stream, reproducible from MASTER_SEED.

    Passing a list as the seed gives numpy a distinct stream per `stream` value. We use one
    stream per purpose (drive, plotting probe, each test) so that adding a random draw in one
    place does not shift the numbers everywhere else.
    """
    return np.random.default_rng([MASTER_SEED, stream])

print("numpy", np.__version__)

## Test framework

A small test registry, so the notebook can check its own work. `@test("Task 1")` registers a
function; `run_tests("Task 1")` runs the ones in that group and prints a pass/fail line for
each. The runner raises at the end if anything failed, which means a failing test stops the
notebook instead of quietly printing a red line that is easy to scroll past.

Tests that involve random numbers use their own fixed seed, so they give the same answer every
run. A test that passes only sometimes is worse than no test at all.

In [ ]:
TESTS = []   # list of (group, readable name, function)

def test(group):
    """Decorator that registers a test function under a named group."""
    def register(fn):
        # Turn a function name like `test_full_circle` into "full circle" for the printout.
        readable = fn.__name__.removeprefix("test_").replace("_", " ")
        TESTS.append((group, readable, fn))
        return fn
    return register


def run_tests(group=None, verbose=True):
    """Run every registered test (or just one group) and summarise the result.

    Raises AssertionError at the end if any test failed, so a broken notebook stops here
    rather than carrying on with bad numbers.
    """
    selected = [t for t in TESTS if group is None or t[0] == group]
    if not selected:
        raise ValueError(f"no tests registered for group {group!r}")

    failures = []
    for grp, name, fn in selected:
        try:
            fn()
            if verbose:
                print(f"  pass    {name}")
        except AssertionError as exc:
            failures.append((name, str(exc) or "assertion failed"))
            print(f"  FAIL    {name}: {exc}")
        except Exception as exc:                      # a crash is a failure too
            failures.append((name, f"{type(exc).__name__}: {exc}"))
            print(f"  ERROR   {name}: {type(exc).__name__}: {exc}")

    label = group if group else "all groups"
    print(f"\n{len(selected) - len(failures)}/{len(selected)} tests passed ({label})")
    if failures:
        raise AssertionError(f"{len(failures)} test(s) failed: "
                             + "; ".join(n for n, _ in failures))
    return len(selected)


def close(a, b, tol=1e-9):
    """Shorthand for comparing floats and arrays inside tests."""
    return np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=tol)


def poses_close(a, b, tol=1e-9):
    """Compare two poses, wrapping the heading difference before checking it."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return close(a[:2], b[:2], tol) and abs(wrap_to_pi(a[2] - b[2])) < tol


print("test framework ready")

---
# Task 1: Kinematic Motion Model

## 1.1 The model

We use the **unicycle** model with a **differential drive** layer bolted on top, so the same
code takes either a body twist $(v, \omega)$ or a pair of wheel speeds $(\omega_L, \omega_R)$.

The state is the planar pose $\mathbf{x} = (x, y, \theta)$, and the continuous motion is

$$\dot x = v\cos\theta, \qquad \dot y = v\sin\theta, \qquad \dot\theta = \omega .$$

### Why not just use Euler integration

The obvious thing to write is `x += v*cos(theta)*dt`, but the heading is changing during the
step, so that line uses a heading which is already out of date by the end of the step. The
robot always ends up slightly short of where it should be on any turn, and the error is
systematic rather than random, so it builds up rather than averaging out.

If $(v, \omega)$ is constant across the step, the exact answer is easy to write down. The robot
travels along a circular arc of radius $R = v/\omega$ around the instantaneous centre of
curvature, which gives

$$
\theta' = \theta + \omega\,\Delta t, \qquad
x' = x + R\,(\sin\theta' - \sin\theta), \qquad
y' = y - R\,(\cos\theta' - \cos\theta).
$$

This blows up as $\omega \to 0$, since $R \to \infty$, so we fall back to the straight line
formula below a small threshold. The two agree in the limit, and the arc form is exact
everywhere else. Section 1.3 compares the two so the difference is visible rather than just
asserted. We keep the Euler version in the notebook only for that comparison.

Getting this right matters later. Task 4 has to explain the difference between where the robot
thinks it is and where it actually is, and attribute that difference to sensor noise. If the
integrator leaked its own error into the prediction step, the filter would be trying to correct
for a bug in our own code.

### Differential drive layer

With wheel radius $r$ and track width $L$ (the distance between the two wheels):

$$v = \frac{r(\omega_R + \omega_L)}{2}, \qquad \omega = \frac{r(\omega_R - \omega_L)}{L}$$

which rearranges to $\omega_R = (2v + \omega L)/2r$ and $\omega_L = (2v - \omega L)/2r$. The
later tasks all command $(v, \omega)$ directly and only use this layer to enforce a speed limit
on each wheel.

In [ ]:
def wrap_to_pi(a):
    """Wrap an angle (or array of angles) into (-pi, pi].

    Used anywhere an angle is stored or subtracted. Without it, a heading of 359 degrees and a
    heading of -1 degree look 360 degrees apart instead of 2, which breaks error terms.
    """
    return (np.asarray(a) + np.pi) % (2 * np.pi) - np.pi


def unicycle_step(pose, v, w, dt, straight_eps=1e-9):
    """Advance the pose by one step of constant (v, w), integrating the arc exactly.

    pose : (3,) array of (x, y, theta) in metres and radians
    v    : forward speed [m/s]
    w    : turn rate [rad/s]
    dt   : step length [s]

    Returns a new (3,) pose. The input is not modified.
    """
    x, y, th = pose

    if abs(w) < straight_eps:
        # Straight line branch. The arc formula divides by w, so we cannot use it here.
        return np.array([x + v * np.cos(th) * dt,
                         y + v * np.sin(th) * dt,
                         wrap_to_pi(th)])

    # Arc branch. R is signed: positive w turns left, negative w turns right.
    R = v / w
    th_new = th + w * dt
    return np.array([x + R * (np.sin(th_new) - np.sin(th)),
                     y - R * (np.cos(th_new) - np.cos(th)),
                     wrap_to_pi(th_new)])


def unicycle_step_euler(pose, v, w, dt):
    """Naive forward Euler step, kept only as the comparison baseline in section 1.3.

    This is what you get if you evaluate the velocities at the start of the step and hold them
    fixed. Nothing else in the project uses it.
    """
    x, y, th = pose
    return np.array([x + v * np.cos(th) * dt,
                     y + v * np.sin(th) * dt,
                     wrap_to_pi(th + w * dt)])


def simulate(pose0, controls, dt, step_fn=unicycle_step):
    """Roll a list of (v, w) commands forward into a trajectory.

    Returns an (N+1, 3) array of poses including the starting pose, so row k is the pose after
    k commands have been applied.
    """
    poses = [np.asarray(pose0, dtype=float)]
    for v, w in controls:
        poses.append(step_fn(poses[-1], v, w, dt))
    return np.array(poses)


def const_twist(v, w, duration, dt):
    """Hold (v, w) for `duration` seconds, as a list of commands."""
    return [(v, w)] * int(round(duration / dt))


@dataclass
class DiffDrive:
    """Differential drive wheel layer sitting on top of the unicycle body twist."""
    wheel_radius: float = 0.05     # r [m]
    track_width: float = 0.30      # L [m], distance between the two wheels
    max_wheel_rate: float = 12.0   # [rad/s], about 0.6 m/s at r = 0.05

    def wheels_to_twist(self, w_left, w_right):
        """Wheel angular speeds [rad/s] to body twist (v [m/s], w [rad/s])."""
        v = self.wheel_radius * (w_right + w_left) / 2.0
        w = self.wheel_radius * (w_right - w_left) / self.track_width
        return v, w

    def twist_to_wheels(self, v, w):
        """Body twist to wheel angular speeds. The inverse of wheels_to_twist."""
        w_right = (2 * v + w * self.track_width) / (2 * self.wheel_radius)
        w_left = (2 * v - w * self.track_width) / (2 * self.wheel_radius)
        return w_left, w_right

    def clamp_twist(self, v, w):
        """Scale a twist down until both wheels are inside their speed limit.

        We scale v and w by the same factor rather than clipping them separately. The path
        curvature is R = v/w, so scaling both leaves R unchanged and the robot follows the same
        arc, just more slowly. Clipping them separately would bend the path instead.
        """
        wl, wr = self.twist_to_wheels(v, w)
        peak = max(abs(wl), abs(wr))
        if peak > self.max_wheel_rate:
            s = self.max_wheel_rate / peak
            return v * s, w * s
        return v, w


ROBOT = DiffDrive()
print("twist -> wheels -> twist round trip:", ROBOT.wheels_to_twist(*ROBOT.twist_to_wheels(0.4, 0.7)))

## 1.2 Checking it against cases we can do by hand

Four cases where the answer can be worked out on paper. All of them start from the origin pose
$(0, 0, 0)$ and hold one twist, so the closed form is

- $\omega = 0$: a straight line ending at $(vT,\, 0,\, 0)$.
- $\omega \ne 0$: an arc of radius $R = v/\omega$ swept through $\phi = \omega T$, ending at
  $\big(R\sin\phi,\; R(1-\cos\phi),\; \phi\big)$.

| # | Command | Duration | Worked out by hand |
|---|---|---|---|
| 1 | $v=0.5$, $\omega=0$ | 4 s | drives 2 m along $+x$, giving $(2, 0, 0)$ |
| 2 | $v=0$, $\omega=\pi/4$ | 2 s | turns in place through 90 degrees, giving $(0, 0, \pi/2)$ |
| 3 | $v=1.0$, $\omega=0.5$ | $\pi$ s | a quarter of a circle of radius 2, giving $(2, 2, \pi/2)$ |
| 4 | $v=1.0$, $\omega=0.5$ | $4\pi$ s | one full circle, so back to $(0, 0, 0)$ |

Case 4 is the most useful of the four. Any per step bias piles up around the loop instead of
cancelling, so coming back to the starting point to machine precision is a real check rather
than a formality.

Each case is integrated in **1000 steps**, so what is being tested is the repeated integration
and not a single evaluation of a formula. We derive the step size as $\Delta t = T/1000$ rather
than fixing it at, say, 0.01 s. Two of the durations are irrational, so a round step size would
not divide them evenly, and the simulation would run a slightly shorter arc than the hand
calculation assumes. That mismatch would show up as an integration error when it is really just
a bookkeeping error. The expected poses in the code below are the literal values from the table
above, not values the code recomputes for itself.

In [ ]:
def analytic_const_twist(v, w, T, straight_eps=1e-9):
    """Closed form pose after holding (v, w) for T seconds, starting from the origin pose."""
    if abs(w) < straight_eps:
        return np.array([v * T, 0.0, 0.0])
    R, phi = v / w, w * T
    return np.array([R * np.sin(phi), R * (1 - np.cos(phi)), wrap_to_pi(phi)])


N_STEPS = 1000   # steps per case, so we are testing repeated integration

#        name               v     w          T          pose worked out by hand
CASES = [("straight 2 m",   0.5,  0.0,       4.0,       [2.0, 0.0, 0.0]),
         ("turn 90 in place", 0.0, np.pi / 4, 2.0,      [0.0, 0.0, np.pi / 2]),
         ("quarter circle",  1.0,  0.5,       np.pi,     [2.0, 2.0, np.pi / 2]),
         ("full circle",     1.0,  0.5,       4 * np.pi, [0.0, 0.0, 0.0])]

print(f"{'case':<19}{'simulated (x, y, theta)':<34}{'expected by hand':<34}{'max err':>10}")
print("-" * 97)

trajectories = {}
for name, v, w, T, expected in CASES:
    dt = T / N_STEPS
    traj = simulate([0, 0, 0], [(v, w)] * N_STEPS, dt)
    got, want = traj[-1], np.array(expected)

    # Compare position directly and heading through wrap_to_pi, so that a result of -1e-16
    # radians is not counted as being 2*pi away from +1e-16.
    err = np.max(np.abs(np.r_[got[:2] - want[:2], wrap_to_pi(got[2] - want[2])]))
    trajectories[name] = traj

    fmt = lambda p: f"({p[0]:+7.4f}, {p[1]:+7.4f}, {p[2]:+7.4f})"
    print(f"{name:<19}{fmt(got):<34}{fmt(want):<34}{err:>10.2e}")

print("\nSee the Task 1 test block below for these cases as assertions.")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.3))
for ax, (name, v, w, T, _) in zip(axes, CASES):
    traj = trajectories[name]
    ax.plot(traj[:, 0], traj[:, 1], lw=2, color="#2b6cb0")
    ax.plot(*traj[0, :2], "o", color="#38a169", ms=7, label="start")
    ax.plot(*traj[-1, :2], "*", color="#c53030", ms=13, label="end")

    # Draw a few heading arrows so the direction of travel is visible, not just the shape.
    for k in np.linspace(0, len(traj) - 1, 8, dtype=int):
        x, y, th = traj[k]
        ax.arrow(x, y, 0.16 * np.cos(th), 0.16 * np.sin(th),
                 head_width=0.07, color="#718096", lw=0.8, alpha=0.8)

    ax.set_title(f"{name}\nv={v:.1f}, w={w:.2f}, T={T:.2f}s", fontsize=8)
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]")

axes[0].set_ylabel("y [m]")
axes[0].legend(fontsize=7, loc="best")
fig.suptitle("Task 1: the four hand checked cases", y=1.06, fontsize=11)
plt.tight_layout(); plt.show()

## 1.3 Exact arc versus Euler

The same four cases run through both integrators at a range of step sizes. For a constant
twist the arc formula is exact, so its error sits at machine precision no matter how coarse the
step gets. Euler's error is proportional to the step size, and it is a consistent under turn
rather than random scatter, which is why the full circle case never quite closes.

At the 0.05 s step this project uses, Euler would be about 3.5 cm out over these short test
runs. That is the same order as the sensor noise we are about to add in Task 2, which would
make it genuinely hard to tell modelling error apart from sensor error.

In [ ]:
dts = [0.5, 0.2, 0.05, 0.01]
rows = []

for dt in dts:
    row = [dt]
    for step_fn in (unicycle_step, unicycle_step_euler):
        worst = 0.0
        for _, v, w, T, _ in CASES:
            n = int(round(T / dt))
            # Compare against the closed form at the duration actually simulated (n*dt) so
            # that rounding the duration to a whole number of steps is not counted as error.
            got = simulate([0, 0, 0], [(v, w)] * n, dt, step_fn)[-1]
            want = analytic_const_twist(v, w, n * dt)
            worst = max(worst, np.linalg.norm(got[:2] - want[:2]))
        row.append(worst)
    rows.append(row)

print(f"{'dt [s]':>8}{'exact arc error [m]':>24}{'Euler error [m]':>20}")
print("-" * 52)
for dt, e_exact, e_euler in rows:
    print(f"{dt:>8.2f}{e_exact:>24.2e}{e_euler:>20.2e}")

fig, ax = plt.subplots(figsize=(5.2, 3.2))
r = np.array(rows)
# Clamp the exact arc curve away from zero so it can be drawn on a log axis.
ax.loglog(r[:, 0], np.maximum(r[:, 1], 1e-17), "o-", label="exact arc (what we use)")
ax.loglog(r[:, 0], r[:, 2], "s--", label="forward Euler")
ax.set_xlabel("step size dt [s]")
ax.set_ylabel("worst final position error [m]")
ax.set_title("Integrator accuracy across the 4 test cases")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 1.4 Task 1 tests

Beyond the four hand checks, most of these test properties the model should have regardless of
the particular numbers: that splitting a command in half changes nothing, that starting
somewhere else just shifts the whole path, that the robot stays a constant distance from the
centre of its turn, and so on. Those catch sign errors and mixed up arguments that a single
worked example can miss.

In [ ]:
# ---------------------------------------------------------------- angles
@test("Task 1")
def test_wrap_to_pi_on_known_angles():
    assert close(wrap_to_pi(0.0), 0.0)
    assert close(wrap_to_pi(np.pi), np.pi)               # +pi stays put
    assert close(wrap_to_pi(-np.pi), np.pi)              # -pi maps onto +pi
    assert close(wrap_to_pi(3 * np.pi), np.pi)
    assert close(wrap_to_pi(2 * np.pi + 0.3), 0.3)
    assert close(wrap_to_pi(-2 * np.pi - 0.3), -0.3)


@test("Task 1")
def test_wrap_to_pi_always_lands_in_range():
    a = make_rng(1001).uniform(-100, 100, 5000)
    wrapped = wrap_to_pi(a)
    assert np.all(wrapped > -np.pi - 1e-12) and np.all(wrapped <= np.pi + 1e-12)
    # Wrapping must only change the angle by whole turns.
    turns = (a - wrapped) / (2 * np.pi)
    assert close(turns, np.round(turns), 1e-9)


# --------------------------------------------------- the four hand checks
@test("Task 1")
def test_straight_line_matches_hand_calculation():
    got = simulate([0, 0, 0], [(0.5, 0.0)] * 1000, 4.0 / 1000)[-1]
    assert poses_close(got, [2.0, 0.0, 0.0]), got


@test("Task 1")
def test_turning_in_place_matches_hand_calculation():
    got = simulate([0, 0, 0], [(0.0, np.pi / 4)] * 1000, 2.0 / 1000)[-1]
    assert poses_close(got, [0.0, 0.0, np.pi / 2]), got


@test("Task 1")
def test_quarter_circle_matches_hand_calculation():
    got = simulate([0, 0, 0], [(1.0, 0.5)] * 1000, np.pi / 1000)[-1]
    assert poses_close(got, [2.0, 2.0, np.pi / 2]), got


@test("Task 1")
def test_full_circle_returns_to_the_start():
    got = simulate([0, 0, 0], [(1.0, 0.5)] * 1000, 4 * np.pi / 1000)[-1]
    assert poses_close(got, [0.0, 0.0, 0.0]), got


# ------------------------------------------------- properties of the model
@test("Task 1")
def test_a_zero_command_does_nothing():
    p = np.array([1.3, -0.7, 0.9])
    assert poses_close(unicycle_step(p, 0.0, 0.0, 0.1), p)


@test("Task 1")
def test_step_size_does_not_change_the_answer():
    # For a constant twist the arc form is exact, so 10 steps and 10000 steps must agree.
    coarse = simulate([0, 0, 0], [(0.8, 0.4)] * 10, 3.0 / 10)[-1]
    fine = simulate([0, 0, 0], [(0.8, 0.4)] * 10000, 3.0 / 10000)[-1]
    assert poses_close(coarse, fine, 1e-9), (coarse, fine)


@test("Task 1")
def test_the_two_branches_agree_near_zero_turn_rate():
    # Just above the threshold we take the arc branch, just below it the straight branch.
    # If they disagreed there would be a jump in behaviour at the crossover.
    p = np.array([0.4, 1.1, 0.6])
    arc = unicycle_step(p, 0.5, 1e-8, 0.05)
    straight = unicycle_step(p, 0.5, 0.0, 0.05)
    assert poses_close(arc, straight, 1e-9)


@test("Task 1")
def test_negative_speed_reverses_along_the_heading():
    p = np.array([2.0, 1.0, 0.0])
    assert poses_close(unicycle_step(p, -0.5, 0.0, 2.0), [1.0, 1.0, 0.0])


@test("Task 1")
def test_negative_turn_rate_turns_clockwise():
    got = unicycle_step([0, 0, 0], 0.0, -1.0, 1.0)
    assert got[2] < 0, got
    # And a left turn of the same size should mirror it exactly.
    left = simulate([0, 0, 0], [(0.6, 0.8)] * 200, 0.01)[-1]
    right = simulate([0, 0, 0], [(0.6, -0.8)] * 200, 0.01)[-1]
    assert close(left[0], right[0]) and close(left[1], -right[1])


@test("Task 1")
def test_distance_from_the_turn_centre_stays_constant():
    # Driving a constant twist traces a circle, so every pose on it is the same distance R
    # from the centre of curvature. The centre sits at R to the left of the starting heading.
    v, w = 0.7, 0.55
    R = v / w
    traj = simulate([0, 0, 0], [(v, w)] * 500, 0.01)
    centre = np.array([0.0, R])
    radii = np.linalg.norm(traj[:, :2] - centre, axis=1)
    assert close(radii, R, 1e-9), (radii.min(), radii.max())


@test("Task 1")
def test_splitting_a_command_in_half_changes_nothing():
    p = np.array([1.0, 2.0, 0.3])
    once = unicycle_step(p, 0.6, 0.4, 0.2)
    twice = unicycle_step(unicycle_step(p, 0.6, 0.4, 0.1), 0.6, 0.4, 0.1)
    assert poses_close(once, twice)


@test("Task 1")
def test_starting_somewhere_else_just_shifts_the_path():
    cmds = [(0.5, 0.3)] * 200
    base = simulate([0, 0, 0], cmds, 0.02)
    shifted = simulate([3.0, -1.5, 0.0], cmds, 0.02)
    assert close(shifted[:, :2] - base[:, :2], np.array([3.0, -1.5]))
    assert close(shifted[:, 2], base[:, 2])


@test("Task 1")
def test_starting_at_a_different_heading_rotates_the_path():
    cmds = [(0.5, 0.3)] * 200
    a = np.deg2rad(37.0)
    base = simulate([0, 0, 0], cmds, 0.02)
    rotated = simulate([0, 0, a], cmds, 0.02)
    R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    assert close(rotated[:, :2], base[:, :2] @ R.T, 1e-9)
    assert close(wrap_to_pi(rotated[:, 2] - base[:, 2] - a), 0.0, 1e-9)


@test("Task 1")
def test_distance_travelled_matches_speed_times_time():
    v, T, n = 0.45, 6.0, 600
    traj = simulate([0, 0, 0], [(v, 0.35)] * n, T / n)
    length = np.sum(np.linalg.norm(np.diff(traj[:, :2], axis=0), axis=1))
    # The path is made of straight chords across each arc, so the measured length is a hair
    # short of the true arc length. A tenth of a percent is plenty of room.
    assert abs(length - v * T) < 1e-3 * v * T, length


@test("Task 1")
def test_euler_error_shrinks_as_the_step_shrinks():
    def euler_error(dt):
        n = int(round(3.0 / dt))
        got = simulate([0, 0, 0], [(1.0, 0.6)] * n, dt, unicycle_step_euler)[-1]
        return np.linalg.norm(got[:2] - analytic_const_twist(1.0, 0.6, n * dt)[:2])
    coarse, fine = euler_error(0.2), euler_error(0.02)
    assert fine < coarse / 5, (coarse, fine)     # roughly first order, so about 10x better


# ------------------------------------------------------- differential drive
@test("Task 1")
def test_wheel_speeds_round_trip():
    for v, w in [(0.4, 0.7), (0.0, 1.2), (-0.3, 0.0), (0.55, -0.9)]:
        assert close(ROBOT.wheels_to_twist(*ROBOT.twist_to_wheels(v, w)), (v, w))


@test("Task 1")
def test_equal_wheel_speeds_drive_straight():
    v, w = ROBOT.wheels_to_twist(5.0, 5.0)
    assert close(w, 0.0) and close(v, ROBOT.wheel_radius * 5.0)


@test("Task 1")
def test_opposite_wheel_speeds_spin_on_the_spot():
    v, w = ROBOT.wheels_to_twist(-4.0, 4.0)
    assert close(v, 0.0) and w > 0


@test("Task 1")
def test_clamping_keeps_both_wheels_inside_the_limit():
    for v, w in [(3.0, 0.0), (0.0, 9.0), (2.5, -4.0)]:
        wl, wr = ROBOT.twist_to_wheels(*ROBOT.clamp_twist(v, w))
        assert max(abs(wl), abs(wr)) <= ROBOT.max_wheel_rate + 1e-9


@test("Task 1")
def test_clamping_keeps_the_path_curvature():
    v, w = 3.0, 1.5
    vc, wc = ROBOT.clamp_twist(v, w)
    assert vc < v                            # it really was clamped
    assert close(vc / wc, v / w, 1e-9)       # same radius of curvature


@test("Task 1")
def test_clamping_leaves_slow_commands_alone():
    assert close(ROBOT.clamp_twist(0.2, 0.3), (0.2, 0.3))


run_tests("Task 1")

---
# Task 2: Environment and Sensor Simulation

Three pieces, all driven by one `NoiseParams` object:

1. **The room.** An 8 m by 6 m household space with furniture as rectangular obstacles, a
   kitchen counter and partition wall that leave a single 1.2 m doorway, and 5 landmarks.
2. **Noisy odometry.** Noise goes in at the velocity level, so the error enters through the
   commands and then compounds through the integrator, which is how wheel slip actually behaves.
3. **Noisy sensing of the outside world.** Range and bearing to landmarks, which Task 4's
   particle filter will use, and a 180 beam lidar, which Task 5 will use to build the map.

## 2.1 The room

The layout is deliberately awkward. The counter and the partition split the room into a living
area on the left and a kitchen on the right, joined by one doorway. A straight line from the
start pose to the goal runs into the partition, so Task 3's planner has real work to do rather
than just drawing a line. The doorway also creates a stretch where landmark visibility is poor,
which is exactly the situation that makes Task 4 interesting.

Two targets are marked for the arm. The **light switch** on the right hand wall at $x = 7.98$ m,
$y = 4.8$ m sits 1.02 m off the floor, and is the target for the switch press. The **person**
seated near the dining table is the target for the handover.

In [ ]:
@dataclass
class Rect:
    """An axis aligned rectangular obstacle, given by its lower left corner and its size.

    `full_height` separates obstacles that block both sensors from low furniture that only
    blocks the lidar. On a real base the lidar sits low on the chassis while the landmark
    camera is up on a mast at roughly 1.2 m, so a coffee table hides the wall behind it from
    the lidar but not a marker mounted high up. Only the partition wall and the bookshelf are
    tall enough here to block both.
    """
    x: float
    y: float
    w: float
    h: float
    name: str = ""
    full_height: bool = False

    @property
    def bounds(self):
        """(xmin, ymin, xmax, ymax)."""
        return self.x, self.y, self.x + self.w, self.y + self.h

    def contains(self, pt, margin=0.0):
        """True if `pt` is inside the rectangle, optionally grown by `margin` on each side."""
        x0, y0, x1, y1 = self.bounds
        return (x0 - margin <= pt[0] <= x1 + margin) and (y0 - margin <= pt[1] <= y1 + margin)

    def distance_to(self, pt):
        """Shortest distance from `pt` to the rectangle. Zero if the point is inside it.

        This is the right test for a round robot: it collides exactly when this distance is
        less than its radius. Growing the box by the radius instead would be too strict near
        the corners, because it would flag a point that is diagonally clear of the corner.
        """
        x0, y0, x1, y1 = self.bounds
        dx = max(x0 - pt[0], 0.0, pt[0] - x1)
        dy = max(y0 - pt[1], 0.0, pt[1] - y1)
        return float(np.hypot(dx, dy))


@dataclass
class Room:
    """The world: its extent, its furniture, its landmarks and the arm's two targets."""
    width: float
    height: float
    furniture: List[Rect]
    landmarks: np.ndarray             # (N, 2) positions of the range and bearing beacons
    switch_xy: Tuple[float, float]    # wall switch, the target for the switch press
    switch_z: float                   # its height off the floor [m]
    person_xy: Tuple[float, float]    # seated person, the target for the handover
    person_z: float

    def clearance(self, pt):
        """Distance from `pt` to the nearest obstacle or wall. Negative outside the room."""
        to_wall = min(pt[0], self.width - pt[0], pt[1], self.height - pt[1])
        to_furniture = min((f.distance_to(pt) for f in self.furniture), default=np.inf)
        return min(to_wall, to_furniture)

    def is_free(self, pt, margin=0.0):
        """True if a disc of radius `margin` centred on `pt` fits without touching anything."""
        return self.clearance(pt) >= margin


def build_room() -> Room:
    """The household layout. All dimensions in metres."""
    furniture = [
        Rect(0.60, 4.20, 2.40, 0.90, "sofa"),
        Rect(1.20, 2.70, 1.20, 0.70, "coffee table"),
        Rect(0.30, 0.30, 1.70, 0.50, "tv stand"),
        Rect(3.40, 0.00, 0.55, 2.40, "kitchen counter"),
        Rect(3.40, 3.60, 0.55, 2.40, "partition wall", full_height=True),
        Rect(4.90, 3.40, 1.80, 1.00, "dining table"),
        Rect(5.10, 2.60, 0.50, 0.50, "chair A"),
        Rect(6.20, 2.60, 0.50, 0.50, "chair B"),
        Rect(7.35, 0.80, 0.55, 2.00, "bookshelf", full_height=True),
    ]
    # The counter runs up to y = 2.40 and the partition starts at y = 3.60, so the gap between
    # them is the 1.2 m doorway, the only way through.

    landmarks = np.array([
        [0.25, 5.75],   # L0, far corner of the living room
        [0.25, 0.25],   # L1, near corner of the living room
        [3.72, 2.40],   # L2, corner of the counter, right at the doorway
        [7.75, 0.25],   # L3, near corner of the kitchen
        [7.75, 5.75],   # L4, far corner of the kitchen, the closest one to the goal
    ])

    return Room(width=8.0, height=6.0, furniture=furniture, landmarks=landmarks,
                switch_xy=(7.98, 4.80), switch_z=1.02,
                person_xy=(5.80, 4.75), person_z=0.95)


ROOM = build_room()
ROBOT_RADIUS = 0.22    # the base is a 0.44 m disc, used for every clearance check

# The poses every later task works between: start in the living room, finish square on to the
# kitchen wall with the switch in front of the robot.
START_POSE = np.array([1.00, 1.60, 0.0])
GOAL_POSE = np.array([7.05, 4.80, 0.0])

print(f"Room {ROOM.width} x {ROOM.height} m, {len(ROOM.furniture)} obstacles, "
      f"{len(ROOM.landmarks)} landmarks")
print(f"start clearance {ROOM.clearance(START_POSE[:2]):.2f} m, "
      f"goal clearance {ROOM.clearance(GOAL_POSE[:2]):.2f} m "
      f"(robot radius {ROBOT_RADIUS} m)")

In [ ]:
def draw_room(ax, room=ROOM, landmarks=True, targets=True, labels=False):
    """Draw the room, its furniture and optionally the landmarks and targets.

    Reused by every figure in the notebook so that all the maps look the same.
    """
    ax.add_patch(Rectangle((0, 0), room.width, room.height, fc="#f7fafc",
                           ec="#2d3748", lw=2, zorder=0))
    for f in room.furniture:
        # Tall obstacles get a darker edge, since they are the ones that hide landmarks.
        ax.add_patch(Rectangle((f.x, f.y), f.w, f.h, fc="#a0aec0",
                               ec="#1a202c" if f.full_height else "#4a5568",
                               lw=1.6 if f.full_height else 1.0, zorder=1))
        if labels:
            ax.text(f.x + f.w / 2, f.y + f.h / 2, f.name, ha="center", va="center",
                    fontsize=6, color="#1a202c", zorder=3)

    if landmarks:
        ax.plot(room.landmarks[:, 0], room.landmarks[:, 1], "^", color="#d69e2e",
                ms=9, mec="#744210", zorder=4, label="landmarks")
        for i, (lx, ly) in enumerate(room.landmarks):
            ax.text(lx, ly + 0.18, f"L{i}", ha="center", fontsize=6.5,
                    color="#744210", zorder=4)

    if targets:
        ax.plot(*room.switch_xy, "s", color="#c53030", ms=8, zorder=4, label="light switch")
        ax.plot(*room.person_xy, "P", color="#6b46c1", ms=10, zorder=4, label="person")

    ax.set_xlim(-0.3, room.width + 0.3)
    ax.set_ylim(-0.3, room.height + 0.3)
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")


def draw_robot(ax, pose, color="#2b6cb0", radius=ROBOT_RADIUS, alpha=1.0, zorder=5):
    """Draw the base as a disc with a line showing which way it is facing."""
    x, y, th = pose
    ax.add_patch(Circle((x, y), radius, fc=color, ec="black", lw=0.8,
                        alpha=alpha, zorder=zorder))
    ax.plot([x, x + radius * 1.5 * np.cos(th)], [y, y + radius * 1.5 * np.sin(th)],
            color="black", lw=1.4, alpha=alpha, zorder=zorder + 1)


fig, ax = plt.subplots(figsize=(7.2, 5.6))
draw_room(ax, labels=True)
draw_robot(ax, START_POSE)
draw_robot(ax, GOAL_POSE, color="#38a169")
ax.plot([START_POSE[0], GOAL_POSE[0]], [START_POSE[1], GOAL_POSE[1]], "r--", lw=1.2,
        alpha=0.7, label="straight line to the goal (blocked)")
ax.annotate("doorway\n(1.2 m)", xy=(3.68, 3.0), xytext=(2.3, 0.9), fontsize=7.5,
            arrowprops=dict(arrowstyle="->", lw=1))
ax.set_title("Task 2: the room, its obstacles, the landmarks and the two arm targets")
ax.legend(fontsize=7, loc="upper left", framealpha=0.9)
plt.tight_layout(); plt.show()

### Ground truth occupancy grid

The same room rasterised at 5 cm. The robot never sees this. It is the reference that Task 5's
estimated map gets compared against.

In [ ]:
GRID_RES = 0.05   # metres per cell

def rasterize(room=ROOM, res=GRID_RES):
    """Ground truth occupancy: 1 where a cell centre lands in furniture, 0 elsewhere.

    Indexed [row, col] = [y, x], which is what imshow(origin="lower") expects.
    """
    nx, ny = int(round(room.width / res)), int(round(room.height / res))
    xs = (np.arange(nx) + 0.5) * res      # cell centres, not edges
    ys = (np.arange(ny) + 0.5) * res
    X, Y = np.meshgrid(xs, ys)

    occ = np.zeros((ny, nx), dtype=np.uint8)
    for f in room.furniture:
        x0, y0, x1, y1 = f.bounds
        occ |= ((X >= x0) & (X <= x1) & (Y >= y0) & (Y <= y1)).astype(np.uint8)
    return occ, (nx, ny)


GT_GRID, (NX, NY) = rasterize()

fig, ax = plt.subplots(figsize=(6.0, 4.6))
ax.imshow(GT_GRID, origin="lower", cmap="Greys", extent=[0, ROOM.width, 0, ROOM.height],
          vmin=0, vmax=1)
ax.set_title(f"Ground truth occupancy grid ({NX} by {NY} cells at {GRID_RES} m)")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")

furniture_area = sum(f.w * f.h for f in ROOM.furniture)
print(f"grid {NX} x {NY} = {GT_GRID.size} cells, {GT_GRID.mean() * 100:.1f}% occupied")
print(f"furniture covers {furniture_area:.2f} m^2 of {ROOM.width * ROOM.height:.0f} m^2 "
      f"({100 * furniture_area / (ROOM.width * ROOM.height):.1f}%)")
plt.tight_layout(); plt.show()

## 2.2 Noise settings

Every noise source reads from one object, and a single `scale` multiplier moves all of them
together. Section 2.7 uses that to show the noise really is tunable rather than just present.

### Odometry

The noise goes in at the velocity level, following the velocity motion model in Thrun's
*Probabilistic Robotics*, rather than being added to the finished pose as a random walk:

$$\hat v = v + \varepsilon(\alpha_1 v^2 + \alpha_2 \omega^2), \quad
  \hat\omega = \omega + \varepsilon(\alpha_3 v^2 + \alpha_4 \omega^2), \quad
  \hat\gamma = \varepsilon(\alpha_5 v^2 + \alpha_6 \omega^2)$$

where $\varepsilon(b)$ is a zero mean Gaussian of variance $b$. The corrupted twist then goes
through the same `unicycle_step` from Task 1, and $\hat\gamma$ adds a small extra turn at the
end of the step, which covers the base finishing a step slightly rotated from where the arc
alone would put it.

Two things follow from this, and both matter later. The error is **proportional to the motion**,
so sitting still adds none and turning fast adds the most, and it is **integrated**, so a
heading error rotates every bit of travel that comes after it. Errors therefore grow faster
than linearly with distance. That structure is what makes the error correctable by a filter,
which is the whole point of Task 4.

### Landmarks and lidar

Range noise has a fixed part plus a part proportional to the range, which is how a real time of
flight sensor behaves. Bearing noise is constant. Landmarks are also subject to a detection
probability and to occlusion, so a beacon behind the partition simply is not reported.

In [ ]:
@dataclass
class NoiseParams:
    """Every noise magnitude in the simulator, in one place."""

    # --- odometry, the alphas of the velocity motion model ------------------
    a1: float = 0.06   # noise on v from v^2, i.e. slip while driving
    a2: float = 0.02   # noise on v from w^2
    a3: float = 0.02   # noise on w from v^2, i.e. veering while driving straight
    a4: float = 0.10   # noise on w from w^2, turning is the worst offender
    a5: float = 0.005  # extra end of step rotation, from v^2
    a6: float = 0.010  # extra end of step rotation, from w^2

    # --- landmark range and bearing sensor ---------------------------------
    sigma_r0: float = 0.035              # range noise, fixed part [m]
    sigma_r1: float = 0.012              # range noise, per metre of range [m/m]
    sigma_b: float = np.deg2rad(1.6)     # bearing noise [rad]
    lm_max_range: float = 8.0            # [m], comfortably under the 10 m room diagonal
    lm_fov: float = np.deg2rad(360.0)    # a ring sensor, so it sees all round
    lm_p_detect: float = 0.92            # chance of getting a reading from a visible landmark
    lm_occlusion: bool = True            # tall obstacles block the line of sight

    # --- lidar --------------------------------------------------------------
    n_beams: int = 180
    lidar_max_range: float = 5.0         # [m]
    sigma_lidar: float = 0.02            # range noise [m]
    lidar_p_dropout: float = 0.01        # fraction of beams that return nothing at all

    def scaled(self, s: float) -> "NoiseParams":
        """Return a copy with every noise magnitude multiplied by `s`.

        The alphas are variances, so they scale by s^2 to make the standard deviations scale
        by s. At s = 0 everything is exact, at s = 1 these are the nominal settings.
        """
        return replace(
            self,
            a1=self.a1 * s**2, a2=self.a2 * s**2, a3=self.a3 * s**2,
            a4=self.a4 * s**2, a5=self.a5 * s**2, a6=self.a6 * s**2,
            sigma_r0=self.sigma_r0 * s, sigma_r1=self.sigma_r1 * s, sigma_b=self.sigma_b * s,
            # A perfect sensor never misses, so the miss rate shrinks towards zero with s.
            lm_p_detect=1 - (1 - self.lm_p_detect) * s,
            sigma_lidar=self.sigma_lidar * s,
            lidar_p_dropout=self.lidar_p_dropout * s)


NOISE = NoiseParams()

print("odometry alphas   :", [NOISE.a1, NOISE.a2, NOISE.a3, NOISE.a4, NOISE.a5, NOISE.a6])
print(f"landmark noise    : {NOISE.sigma_r0} m + {NOISE.sigma_r1} m per m of range, "
      f"{np.rad2deg(NOISE.sigma_b):.1f} deg bearing")
print(f"landmark sensor   : {NOISE.lm_max_range} m range, "
      f"{NOISE.lm_p_detect:.0%} detection, occlusion {'on' if NOISE.lm_occlusion else 'off'}")
print(f"lidar             : {NOISE.n_beams} beams, {NOISE.lidar_max_range} m range, "
      f"{NOISE.sigma_lidar} m noise, {NOISE.lidar_p_dropout:.0%} dropout")

In [ ]:
def sample_odometry(v, w, params: NoiseParams, rng):
    """Corrupt a commanded twist with velocity level noise.

    Returns (v_hat, w_hat, gamma_hat), where gamma_hat is an extra turn *rate* applied at the
    end of the step. Note there is no dt here: the noise scales with the commanded speeds, and
    the step length is applied by the caller.
    """
    sd = lambda var: np.sqrt(max(var, 0.0))   # guard against -0.0 from a scale of 0
    v_hat = v + rng.normal(0.0, sd(params.a1 * v**2 + params.a2 * w**2))
    w_hat = w + rng.normal(0.0, sd(params.a3 * v**2 + params.a4 * w**2))
    g_hat = rng.normal(0.0, sd(params.a5 * v**2 + params.a6 * w**2))
    return v_hat, w_hat, g_hat


def odom_step(pose, v, w, dt, params: NoiseParams, rng):
    """One step of dead reckoning: the Task 1 integrator fed a corrupted command."""
    v_hat, w_hat, g_hat = sample_odometry(v, w, params, rng)
    new = unicycle_step(pose, v_hat, w_hat, dt)
    new[2] = wrap_to_pi(new[2] + g_hat * dt)
    return new


# Quick check that turning the noise off recovers Task 1 exactly. There is a proper test for
# this below; this line is here so the failure shows up next to the code that caused it.
_p = np.array([1.0, 2.0, 0.3])
assert poses_close(odom_step(_p, 0.4, 0.3, 0.1, NOISE.scaled(0.0), make_rng(99)),
                   unicycle_step(_p, 0.4, 0.3, 0.1))
print("with noise switched off, odometry reduces to the Task 1 model")

## 2.3 Ray casting

Both outward looking sensors need the same primitive: how far along a ray before it hits
something. Every obstacle here is an axis aligned box, so the slab method gives an exact answer
in closed form. For each axis the ray enters that axis's interval at $t_1$ and leaves it at
$t_2$; the ray hits the box only if the two intervals overlap, that is if
$\max t_{\text{enter}} \le \min t_{\text{exit}}$. No stepping along the ray, no grid, and no
resolution artefacts.

The walls are handled separately, as the distance at which the ray leaves the room's own
bounding box, since the robot is always inside it.

In [ ]:
def ray_box_hit(p, d, box, eps=1e-12):
    """Distance along the ray (p, d) to an axis aligned box, or inf if it misses.

    p is the start point, d is a unit direction, box is (xmin, ymin, xmax, ymax).
    """
    x0, y0, x1, y1 = box
    t_enter, t_exit = -np.inf, np.inf

    for i, (lo, hi) in enumerate(((x0, x1), (y0, y1))):
        if abs(d[i]) < eps:
            # The ray runs parallel to this pair of edges, so it either stays inside the slab
            # forever or never enters it. No t constraint either way.
            if p[i] < lo or p[i] > hi:
                return np.inf
            continue
        t1, t2 = (lo - p[i]) / d[i], (hi - p[i]) / d[i]
        if t1 > t2:
            t1, t2 = t2, t1              # d is negative on this axis, so the order flipped
        t_enter, t_exit = max(t_enter, t1), min(t_exit, t2)
        if t_enter > t_exit:
            return np.inf                # the two slabs never overlap, so it misses

    if t_exit < 0:
        return np.inf                    # the box is entirely behind the ray
    # t_enter <= 0 means we started inside the box, so the first surface met is on the way out.
    return t_enter if t_enter > 0 else t_exit


def ray_room_exit(p, d, room, eps=1e-12):
    """Distance from a point inside the room to the walls, along the ray (p, d)."""
    t_exit = np.inf
    for i, (lo, hi) in enumerate(((0.0, room.width), (0.0, room.height))):
        if abs(d[i]) < eps:
            continue
        # From inside, the wall we hit on this axis is whichever of the two is ahead of us.
        t_exit = min(t_exit, max((lo - p[i]) / d[i], (hi - p[i]) / d[i]))
    return t_exit


def cast_ray(p, angle, room, max_range=np.inf, full_height_only=False):
    """Distance to the nearest obstacle or wall from p along `angle`, capped at max_range.

    full_height_only=True considers only obstacles tall enough to block the mast mounted
    landmark camera. The lidar, which is the default, sees everything.
    """
    d = np.array([np.cos(angle), np.sin(angle)])
    t = ray_room_exit(p, d, room)
    for f in room.furniture:
        if full_height_only and not f.full_height:
            continue
        t = min(t, ray_box_hit(p, d, f.bounds))
    return min(t, max_range)


# Five ranges that can be read straight off the furniture table above.
CAST_CHECKS = [
    # (from, heading [rad], expected range [m], what it hits)
    ((1.0, 1.60), 0.0,       2.40, "front of the kitchen counter at x = 3.40"),
    ((1.0, 1.60), np.pi,     1.00, "left wall at x = 0"),
    ((2.6, 3.20), 0.0,       5.40, "clean line through the doorway to the right wall at x = 8"),
    ((2.6, 3.20), np.pi / 2, 1.00, "near edge of the sofa at y = 4.20"),
    ((7.0, 4.80), 0.0,       1.00, "right wall at x = 8, which is where the switch is mounted"),
]

for p, a, expected, what in CAST_CHECKS:
    got = cast_ray(np.array(p), a, ROOM)
    print(f"  from {str(p):<12} heading {np.rad2deg(a):+6.1f} deg -> {got:5.2f} m   ({what})")
print("\n(these are asserted in the Task 2 test block)")

## 2.4 The two sensors

**Lidar.** `n_beams` spread evenly round the full circle, each one cast, noised, then clipped.
Beams that hit nothing inside `lidar_max_range`, and beams dropped at random, come back as
`inf`. Task 5 has to read that as "free space all the way out to the maximum range" rather than
"an obstacle sitting at the maximum range", which is a classic way to end up with a ring of
phantom walls in the map.

**Landmarks.** For each one: skip it if it is out of range or outside the field of view, skip
it if something tall is in the way, skip it at random with probability $1 - p_{\text{detect}}$,
and otherwise return a noisy $(\text{id}, r, b)$ with the bearing measured relative to the
robot's own heading.

The occlusion test only casts against full height obstacles, for the reason given in the `Rect`
docstring. This is what gives landmark visibility its structure. Going through the doorway
swaps which half of the room's markers are in view, and the bookshelf blanks out the near
kitchen corner just as the robot is closing on the goal.

We return the landmark id with each reading, so correspondence is known. That is a deliberate
simplification. These are meant to be distinguishable beacons rather than anonymous corners,
and it keeps Task 4 about filtering rather than about data association.

In [ ]:
def lidar_scan(pose, room, params: NoiseParams, rng, add_noise=True):
    """Simulated 2D lidar sweep.

    Returns (angles, ranges). The angles are relative to the robot's heading and the ranges are
    inf wherever the beam found nothing within lidar_max_range or was dropped.
    """
    x, y, th = pose
    p = np.array([x, y])
    angles = np.linspace(-np.pi, np.pi, params.n_beams, endpoint=False)
    ranges = np.empty(params.n_beams)

    for i, a in enumerate(angles):
        r = cast_ray(p, th + a, room)      # true distance, uncapped

        if r >= params.lidar_max_range:
            ranges[i] = np.inf             # nothing close enough to see
            continue

        if add_noise:
            if rng.random() < params.lidar_p_dropout:
                ranges[i] = np.inf         # beam lost, e.g. a dark or glancing surface
                continue
            r = r + rng.normal(0.0, params.sigma_lidar)

        ranges[i] = np.clip(r, 0.0, params.lidar_max_range)

    return angles, ranges


def observe_landmarks(pose, room, params: NoiseParams, rng, add_noise=True):
    """Range and bearing readings to the landmarks the robot can currently see.

    Returns a list of (landmark_id, range, bearing), bearing relative to the robot's heading.
    """
    x, y, th = pose
    p = np.array([x, y])
    obs = []

    for i, lm in enumerate(room.landmarks):
        dx, dy = lm - p
        r = float(np.hypot(dx, dy))
        b = wrap_to_pi(np.arctan2(dy, dx) - th)

        if r > params.lm_max_range or abs(b) > params.lm_fov / 2:
            continue

        # Cast towards the landmark and see whether anything tall gets there first. The small
        # tolerance stops a landmark sitting flat against a wall from occluding itself.
        if params.lm_occlusion and cast_ray(p, np.arctan2(dy, dx), room,
                                            full_height_only=True) < r - 1e-6:
            continue

        if add_noise:
            if rng.random() > params.lm_p_detect:
                continue                                   # missed this one
            r = r + rng.normal(0.0, params.sigma_r0 + params.sigma_r1 * r)
            b = wrap_to_pi(b + rng.normal(0.0, params.sigma_b))

        obs.append((i, float(r), float(b)))

    return obs


_clean = observe_landmarks(START_POSE, ROOM, NOISE.scaled(0.0), make_rng(1))
print(f"noise free readings from the start pose ({len(_clean)} of {len(ROOM.landmarks)} visible):")
for i, r, b in _clean:
    print(f"   L{i}: range {r:5.2f} m, bearing {np.rad2deg(b):+7.1f} deg")

## 2.5 A reference drive, and what dead reckoning makes of it

An open loop command sequence takes the base from the living room, through the doorway, past
the dining table and up to the goal in front of the kitchen wall. Each leg is a turn on the
spot followed by a straight run. It is not a clever path, and it is not meant to be. Task 3
replaces the waypoints with a planned path; this exists so that Tasks 2, 4 and 5 have something
to work with before the planner is written.

One detail worth noting: the turn rate and speed for each leg are back solved from a whole
number of steps, so the noise free trajectory lands exactly on each waypoint instead of
overshooting by a fraction of a step. That means we can assert the drive reaches the goal
exactly, and any drift we see afterwards is definitely coming from the noise model rather than
from sloppy command generation.

The same commands are then integrated twice from the same starting pose:

- the **true pose**, using the clean `unicycle_step`, which is the ground truth the simulator
  keeps to itself,
- the **odometry pose**, using `odom_step`, which is what the robot would believe if it only
  had wheel encoders.

The gap between those two is the error Task 4 has to get rid of. It is not a constant offset.
A heading error picked up during the first turn keeps rotating everything that happens after
it, which is why the gap opens up rather than staying put.

In [ ]:
DT = 0.05   # simulation step from here on [s]

# Waypoints chosen by hand: across the living room, through the doorway, along the near side of
# the dining table, then up the right hand side to the goal.
WAYPOINTS = np.array([[2.60, 2.20],
                      [3.67, 3.00],    # middle of the 1.2 m doorway
                      [4.55, 3.00],    # step clear of the counter corner before turning down
                      [4.80, 2.05],
                      [7.00, 2.05],    # clear of both chairs and of the bookshelf
                      [7.05, 4.80]])   # the goal, in front of the switch wall


def waypoint_commands(start, waypoints, final_heading, dt=DT, v_max=0.45, w_max=0.70):
    """Open loop turn then drive commands through a list of waypoints.

    Each leg is a turn on the spot followed by a straight run. Both are quantised to a whole
    number of steps first, then the rate is back solved (w = dtheta / (n * dt)), so the noise
    free path lands on each waypoint exactly rather than approximately.

    Returns (commands, segment_end_times), the second being useful for marking up plots.
    """
    cmds, bounds = [], []
    pose = np.asarray(start, dtype=float)

    def rotate(dtheta):
        nonlocal pose
        n = max(1, int(np.ceil(abs(dtheta) / (w_max * dt))))
        w = dtheta / (n * dt)
        cmds.extend([(0.0, w)] * n)
        for _ in range(n):
            pose = unicycle_step(pose, 0.0, w, dt)
        bounds.append(len(cmds) * dt)

    def advance(dist):
        nonlocal pose
        n = max(1, int(np.ceil(dist / (v_max * dt))))
        v = dist / (n * dt)
        cmds.extend([(v, 0.0)] * n)
        for _ in range(n):
            pose = unicycle_step(pose, v, 0.0, dt)
        bounds.append(len(cmds) * dt)

    for wp in np.asarray(waypoints, dtype=float):
        dx, dy = wp - pose[:2]
        rotate(wrap_to_pi(np.arctan2(dy, dx) - pose[2]))   # face the waypoint
        advance(float(np.hypot(dx, dy)))                   # drive to it
    rotate(wrap_to_pi(final_heading - pose[2]))            # square up at the end

    return cmds, bounds


def run_drive(commands, params: NoiseParams, seed_stream=10, dt=DT, room=ROOM,
              start=START_POSE, scan_every=10):
    """Run the commands, recording the true pose, the odometry pose and the sensor readings.

    scan_every controls how often a full lidar sweep is taken, since sweeping every step is
    slow and a real lidar runs slower than the control loop anyway. Set it very high to skip
    scans entirely, which is what the noise sweep does.
    """
    rng = make_rng(seed_stream)
    true_pose = np.array(start, dtype=float)
    odom_pose = np.array(start, dtype=float)

    log = dict(t=[0.0], true=[true_pose.copy()], odom=[odom_pose.copy()],
               cmds=[], obs=[], scans=[])

    for k, (v, w) in enumerate(commands):
        v, w = ROBOT.clamp_twist(v, w)                     # respect the wheel speed limit
        true_pose = unicycle_step(true_pose, v, w, dt)     # what actually happened
        odom_pose = odom_step(odom_pose, v, w, dt, params, rng)   # what the encoders say

        log["t"].append((k + 1) * dt)
        log["true"].append(true_pose.copy())
        log["odom"].append(odom_pose.copy())
        log["cmds"].append((v, w))
        # Sensors always read against the true pose. The robot does not get to peek at it.
        log["obs"].append(observe_landmarks(true_pose, room, params, rng))
        log["scans"].append(lidar_scan(true_pose, room, params, rng)
                            if k % scan_every == 0 else None)

    for key in ("t", "true", "odom"):
        log[key] = np.array(log[key])
    return log


CMDS, SEG_BOUNDS = waypoint_commands(START_POSE, WAYPOINTS, final_heading=GOAL_POSE[2])
LOG = run_drive(CMDS, NOISE)

final_err = np.linalg.norm(LOG["true"][-1][:2] - LOG["odom"][-1][:2])
head_err = np.rad2deg(abs(wrap_to_pi(LOG["true"][-1][2] - LOG["odom"][-1][2])))
path_len = np.sum(np.linalg.norm(np.diff(LOG["true"][:, :2], axis=0), axis=1))
min_clear = min(ROOM.clearance(p[:2]) for p in LOG["true"])
seen = np.array([len(o) for o in LOG["obs"]])

print(f"drive          : {len(CMDS)} steps, {len(CMDS) * DT:.1f} s, path length {path_len:.2f} m")
print(f"min clearance  : {min_clear:.3f} m (robot radius {ROBOT_RADIUS} m)")
print(f"true final pose: {np.round(LOG['true'][-1], 4)}")
print(f"odom final pose: {np.round(LOG['odom'][-1], 4)}")
print(f"dead reckoning error at the goal: {final_err:.3f} m and {head_err:.1f} deg")
print(f"landmarks seen per step: mean {seen.mean():.2f}, "
      f"none at all on {100 * np.mean(seen == 0):.1f}% of steps")

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 5.8))
draw_room(ax)
ax.plot(LOG["true"][:, 0], LOG["true"][:, 1], color="#2b6cb0", lw=2.2,
        label="true pose", zorder=6)
ax.plot(LOG["odom"][:, 0], LOG["odom"][:, 1], color="#c53030", lw=1.8, ls="--",
        label="odometry only (dead reckoning)", zorder=6)
ax.plot(WAYPOINTS[:, 0], WAYPOINTS[:, 1], "o", mfc="none", mec="#2b6cb0", ms=7,
        label="waypoints", zorder=7)

# Thin lines joining the two paths at the same instant, so the growing gap is easy to see.
for k in range(0, len(LOG["true"]), 25):
    ax.plot([LOG["true"][k, 0], LOG["odom"][k, 0]],
            [LOG["true"][k, 1], LOG["odom"][k, 1]], color="#a0aec0", lw=0.6, zorder=5)

draw_robot(ax, LOG["true"][0])
draw_robot(ax, LOG["true"][-1], color="#38a169")
draw_robot(ax, LOG["odom"][-1], color="#c53030", alpha=0.45)
ax.set_title(f"Task 2: odometry drift over a {path_len:.1f} m drive "
             f"(final error {final_err:.2f} m, {head_err:.1f} deg)")
ax.legend(fontsize=7.5, loc="lower left", framealpha=0.9)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.0))
err_xy = np.linalg.norm(LOG["true"][:, :2] - LOG["odom"][:, :2], axis=1)
err_th = np.rad2deg(np.abs(wrap_to_pi(LOG["true"][:, 2] - LOG["odom"][:, 2])))
axes[0].plot(LOG["t"], err_xy, color="#c53030"); axes[0].set_ylabel("position error [m]")
axes[1].plot(LOG["t"], err_th, color="#805ad5"); axes[1].set_ylabel("heading error [deg]")
for a in axes:
    a.set_xlabel("time [s]")
    for ts in SEG_BOUNDS:
        a.axvline(ts, color="#cbd5e0", lw=0.7, zorder=0)    # turn / drive boundaries
axes[0].set_title("Dead reckoning error over the run (grey lines mark each turn or drive)",
                  loc="left")
plt.tight_layout(); plt.show()

### What the sensors actually see

On the left, one full lidar sweep from the goal pose drawn as rays in the room. The beams stop
on the real furniture and wall surfaces, and those endpoints are what Task 5 will turn into
occupied cells. The gold lines are the landmarks the robot can see from there.

On the right, the same sweep in the robot's own frame. The beams that returned nothing are
drawn in grey out at the maximum range, as a reminder that they are not obstacles.

In [ ]:
scan_pose = LOG["true"][-1]
angles, ranges = lidar_scan(scan_pose, ROOM, NOISE, make_rng(21))
hit = np.isfinite(ranges)

fig = plt.figure(figsize=(12, 4.6))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2, projection="polar")

draw_room(ax1)
px, py, pth = scan_pose
for a, r in zip(angles[hit], ranges[hit]):
    ax1.plot([px, px + r * np.cos(pth + a)], [py, py + r * np.sin(pth + a)],
             color="#e53e3e", lw=0.4, alpha=0.45, zorder=5)
ax1.plot(px + ranges[hit] * np.cos(pth + angles[hit]),
         py + ranges[hit] * np.sin(pth + angles[hit]),
         ".", color="#742a2a", ms=2.5, zorder=6, label="beam endpoints")
for i, r, b in LOG["obs"][-1]:
    ax1.plot([px, px + r * np.cos(pth + b)], [py, py + r * np.sin(pth + b)],
             color="#d69e2e", lw=1.6, zorder=7)
draw_robot(ax1, scan_pose)
ax1.set_title(f"Lidar sweep and landmark sightings at the goal\n"
              f"({hit.sum()} of {len(ranges)} beams came back)")

ax2.plot(angles[hit], ranges[hit], ".", ms=3, color="#2b6cb0", label="hit")
ax2.plot(angles[~hit], np.full((~hit).sum(), NOISE.lidar_max_range), ".", ms=3,
         color="#cbd5e0", label="no return")
ax2.set_rmax(NOISE.lidar_max_range * 1.05)
ax2.set_title("The same sweep in the robot frame\n(0 degrees is straight ahead)",
              pad=14, fontsize=9)
ax2.legend(fontsize=7, loc="lower left", bbox_to_anchor=(-0.15, -0.12))
plt.tight_layout(); plt.show()

## 2.6 Measuring the noise we claim to have added

Every sensor is sampled a few thousand times from a fixed pose, and the spread of the errors is
compared against the standard deviations declared in `NoiseParams`. This is the check that the
noise is genuinely being injected, and injected at the size the writeup says it is. It would be
easy to write a noise model that silently does nothing, and the only way to know is to measure
it.

In [ ]:
rng = make_rng(33)
probe = np.array([7.00, 2.05, 0.35])   # a pose on the reference path, well clear of furniture
N_TRIALS = 4000

# Take the noise free reading first, so we have something to measure the errors against.
truth = {i: (r, b) for i, r, b in observe_landmarks(probe, ROOM, NOISE.scaled(0.0), rng)}
assert truth, "the probe pose cannot see any landmarks, pick another one"
lm_id = min(truth)
r_true, b_true = truth[lm_id]

r_err = np.array([o[1] - r_true for _ in range(N_TRIALS)
                  for o in observe_landmarks(probe, ROOM, NOISE, rng) if o[0] == lm_id])
b_err = np.array([wrap_to_pi(o[2] - b_true) for _ in range(N_TRIALS)
                  for o in observe_landmarks(probe, ROOM, NOISE, rng) if o[0] == lm_id])

# For the lidar, use a cut down 8 beam sensor so we can take a lot of samples quickly.
fast_lidar = replace(NOISE, n_beams=8)
lr_true = lidar_scan(probe, ROOM, replace(fast_lidar, sigma_lidar=0.0),
                     rng, add_noise=False)[1][0]
lr_samples = np.array([lidar_scan(probe, ROOM, fast_lidar, rng)[1][0] for _ in range(1500)])
lr_err = lr_samples[np.isfinite(lr_samples)] - lr_true

v_cmd, w_cmd = 0.45, 0.30
odom_samples = np.array([sample_odometry(v_cmd, w_cmd, NOISE, rng) for _ in range(N_TRIALS)])

# What NoiseParams says these spreads should be.
exp_r = NOISE.sigma_r0 + NOISE.sigma_r1 * r_true
exp_v = np.sqrt(NOISE.a1 * v_cmd**2 + NOISE.a2 * w_cmd**2)
exp_w = np.sqrt(NOISE.a3 * v_cmd**2 + NOISE.a4 * w_cmd**2)

print(f"{'quantity':<36}{'specified':>13}{'measured':>13}")
print("-" * 62)
for label, expected, measured in [
        (f"landmark L{lm_id} range [m]", exp_r, r_err.std()),
        ("landmark bearing [deg]", np.rad2deg(NOISE.sigma_b), np.rad2deg(b_err.std())),
        ("lidar range [m]", NOISE.sigma_lidar, lr_err.std()),
        (f"odometry v at v={v_cmd} w={w_cmd} [m/s]", exp_v, odom_samples[:, 0].std()),
        (f"odometry w at v={v_cmd} w={w_cmd} [rad/s]", exp_w, odom_samples[:, 1].std())]:
    print(f"{label:<36}{expected:>13.4f}{measured:>13.4f}")

print(f"\nlandmark L{lm_id} was detected on {len(r_err) / N_TRIALS:.3f} of trials "
      f"(specified {NOISE.lm_p_detect})")

fig, axes = plt.subplots(1, 3, figsize=(11, 2.9))
for ax, data, sigma, title in [
        (axes[0], r_err, exp_r, f"landmark range error\nmeasured sd {r_err.std():.3f} m"),
        (axes[1], np.rad2deg(b_err), np.rad2deg(NOISE.sigma_b),
         f"landmark bearing error\nmeasured sd {np.rad2deg(b_err.std()):.2f} deg"),
        (axes[2], odom_samples[:, 0] - v_cmd, exp_v,
         f"odometry speed error\nmeasured sd {odom_samples[:, 0].std():.3f} m/s")]:
    ax.hist(data, bins=45, density=True, color="#63b3ed", ec="white", lw=0.3)
    xs = np.linspace(data.min(), data.max(), 200)
    ax.plot(xs, np.exp(-0.5 * (xs / sigma)**2) / (sigma * np.sqrt(2 * np.pi)),
            "r-", lw=1.4, label="what we asked for")
    ax.set_title(title, fontsize=8)
    ax.legend(fontsize=6.5)
plt.tight_layout(); plt.show()

## 2.7 Turning the noise up and down

The same drive and the same seeds, with every noise magnitude multiplied by one number. At a
scale of 0 the odometry is exact and the two paths sit on top of each other. Raising the scale
makes dead reckoning worse in a smooth, predictable way. Twelve seeds per setting, so what is
shown is the spread rather than one lucky run.

This is the knob Task 4 will be judged against. The filter has to stay accurate as the scale
goes up, and the gap between the red band here and the filtered estimate is the localization
result.

In [ ]:
SCALES = [0.0, 0.5, 1.0, 2.0]
N_SEEDS = 12

sweep = {}
for s in SCALES:
    params = NOISE.scaled(s)
    errs = []
    for seed in range(N_SEEDS):
        # scan_every is huge here so no lidar sweeps are taken. We only need the poses, and
        # sweeping would make this cell a lot slower for no benefit.
        lg = run_drive(CMDS, params, seed_stream=200 + seed, scan_every=10**6)
        errs.append(np.linalg.norm(lg["true"][:, :2] - lg["odom"][:, :2], axis=1))
    sweep[s] = np.array(errs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(SCALES)))

for (s, errs), c in zip(sweep.items(), colors):
    ax1.fill_between(LOG["t"], errs.min(axis=0), errs.max(axis=0), color=c, alpha=0.18)
    ax1.plot(LOG["t"], np.median(errs, axis=0), color=c, lw=1.8, label=f"noise scale {s:g}")
ax1.set_xlabel("time [s]")
ax1.set_ylabel("dead reckoning error [m]")
ax1.set_title("Error growth against noise scale\n(band is min to max over 12 seeds)", fontsize=9)
ax1.legend(fontsize=7)

draw_room(ax2, landmarks=False, targets=False)
for (s, _), c in zip(sweep.items(), colors):
    for seed in range(4):
        lg = run_drive(CMDS, NOISE.scaled(s), seed_stream=200 + seed, scan_every=10**6)
        ax2.plot(lg["odom"][:, 0], lg["odom"][:, 1], color=c, lw=1.0, alpha=0.85)
ax2.plot(LOG["true"][:, 0], LOG["true"][:, 1], "k--", lw=1.6, label="true path")
ax2.set_title("Dead reckoned paths, 4 seeds per noise scale", fontsize=9)
ax2.legend(fontsize=7, loc="lower left")
plt.tight_layout(); plt.show()

print(f"{'scale':>7}{'median final error [m]':>26}{'worst of 12 [m]':>18}")
print("-" * 51)
for s, errs in sweep.items():
    print(f"{s:>7g}{np.median(errs[:, -1]):>26.3f}{errs[:, -1].max():>18.3f}")

## 2.8 Task 2 tests

These cover the room geometry, the ray caster, both sensors, the odometry noise and the
reference drive. A few are worth calling out.

The occlusion tests check the height rule in both directions: a landmark hidden behind the
partition must not be reported, while a landmark behind the coffee table must be, even though a
lidar beam pointed the same way stops short at the table. Getting that backwards would be easy
and would quietly change how much information Task 4 has to work with.

The statistical tests all use fixed seeds and generous tolerances. They are there to catch
noise that is missing, doubled or attached to the wrong variable, not to check the quality of
numpy's Gaussian sampler.

In [ ]:
# ------------------------------------------------------------ room geometry
@test("Task 2")
def test_start_and_goal_are_clear_of_the_furniture():
    assert ROOM.is_free(START_POSE[:2], margin=ROBOT_RADIUS)
    assert ROOM.is_free(GOAL_POSE[:2], margin=ROBOT_RADIUS)


@test("Task 2")
def test_points_inside_furniture_are_not_free():
    for f in ROOM.furniture:
        centre = (f.x + f.w / 2, f.y + f.h / 2)
        assert not ROOM.is_free(centre), f.name
        assert close(f.distance_to(centre), 0.0)


@test("Task 2")
def test_points_outside_the_room_are_not_free():
    for pt in [(-0.1, 3.0), (8.1, 3.0), (4.0, -0.1), (4.0, 6.1)]:
        assert not ROOM.is_free(pt, margin=0.0), pt


@test("Task 2")
def test_a_bigger_margin_never_makes_a_point_more_free():
    rng = make_rng(2001)
    pts = np.column_stack([rng.uniform(0, ROOM.width, 300), rng.uniform(0, ROOM.height, 300)])
    for pt in pts:
        if ROOM.is_free(pt, margin=0.4):
            assert ROOM.is_free(pt, margin=0.1)     # clear at 0.4 implies clear at 0.1


@test("Task 2")
def test_distance_to_a_rectangle_matches_hand_values():
    r = Rect(1.0, 1.0, 2.0, 1.0)                    # spans x 1 to 3, y 1 to 2
    assert close(r.distance_to((2.0, 1.5)), 0.0)    # inside
    assert close(r.distance_to((4.0, 1.5)), 1.0)    # straight out to the right
    assert close(r.distance_to((2.0, 3.0)), 1.0)    # straight up
    assert close(r.distance_to((4.0, 3.0)), np.hypot(1.0, 1.0))   # diagonally off a corner


@test("Task 2")
def test_the_grid_has_the_expected_shape_and_occupied_fraction():
    assert GT_GRID.shape == (NY, NX) == (120, 160)
    # The furniture does not overlap, so the occupied area is just the sum of the rectangles.
    expected = sum(f.w * f.h for f in ROOM.furniture) / (ROOM.width * ROOM.height)
    assert abs(GT_GRID.mean() - expected) < 0.02, (GT_GRID.mean(), expected)


@test("Task 2")
def test_the_furniture_does_not_overlap():
    # The occupied fraction test above relies on this, so check it rather than assume it.
    for i, a in enumerate(ROOM.furniture):
        for b in ROOM.furniture[i + 1:]:
            ax0, ay0, ax1, ay1 = a.bounds
            bx0, by0, bx1, by1 = b.bounds
            overlaps = (ax0 < bx1 and bx0 < ax1) and (ay0 < by1 and by0 < ay1)
            assert not overlaps, f"{a.name} overlaps {b.name}"


@test("Task 2")
def test_the_doorway_is_wide_enough_for_the_robot():
    # The counter ends at y = 2.40 and the partition starts at y = 3.60.
    gap = 3.60 - 2.40
    assert gap > 2 * ROBOT_RADIUS, gap
    assert ROOM.is_free((3.67, 3.00), margin=ROBOT_RADIUS)


# --------------------------------------------------------------- ray casting
@test("Task 2")
def test_ray_ranges_match_the_hand_computed_values():
    for p, a, expected, what in CAST_CHECKS:
        got = cast_ray(np.array(p), a, ROOM)
        assert abs(got - expected) < 1e-9, f"{what}: got {got}, expected {expected}"


@test("Task 2")
def test_a_ray_never_reaches_further_than_the_room_diagonal():
    diagonal = np.hypot(ROOM.width, ROOM.height)
    rng = make_rng(2002)
    for _ in range(300):
        p = np.array([rng.uniform(0.3, ROOM.width - 0.3), rng.uniform(0.3, ROOM.height - 0.3)])
        if not ROOM.is_free(p):
            continue
        r = cast_ray(p, rng.uniform(-np.pi, np.pi), ROOM)
        assert 0 < r <= diagonal + 1e-9, r


@test("Task 2")
def test_a_box_behind_the_ray_is_not_hit():
    # Box sits to the right, ray points left, so it must miss.
    assert ray_box_hit(np.array([0.0, 0.0]), np.array([-1.0, 0.0]), (1.0, -1.0, 2.0, 1.0)) == np.inf


@test("Task 2")
def test_a_ray_parallel_to_a_box_misses_it():
    # Travelling along +x at y = 5, while the box only spans y from -1 to 1.
    assert ray_box_hit(np.array([0.0, 5.0]), np.array([1.0, 0.0]), (1.0, -1.0, 2.0, 1.0)) == np.inf


@test("Task 2")
def test_casting_towards_a_wall_gives_the_distance_to_it():
    # From the middle of the room, straight up, with nothing in the way at x = 4.3.
    assert close(cast_ray(np.array([4.3, 0.6]), np.pi / 2, ROOM), ROOM.height - 0.6, 1e-9)


# --------------------------------------------------------------------- lidar
@test("Task 2")
def test_the_lidar_returns_the_number_of_beams_asked_for():
    for n in (8, 45, 180):
        angles, ranges = lidar_scan(GOAL_POSE, ROOM, replace(NOISE, n_beams=n), make_rng(2003))
        assert len(angles) == len(ranges) == n


@test("Task 2")
def test_a_noise_free_lidar_agrees_with_direct_ray_casts():
    params = replace(NOISE, n_beams=36)
    angles, ranges = lidar_scan(GOAL_POSE, ROOM, params, make_rng(2004), add_noise=False)
    for a, r in zip(angles, ranges):
        direct = cast_ray(GOAL_POSE[:2], GOAL_POSE[2] + a, ROOM)
        if np.isinf(r):
            assert direct >= params.lidar_max_range      # inf only where nothing is in range
        else:
            assert close(r, direct, 1e-9)


@test("Task 2")
def test_lidar_ranges_stay_inside_their_limits():
    _, ranges = lidar_scan(START_POSE, ROOM, NOISE, make_rng(2005))
    finite = ranges[np.isfinite(ranges)]
    assert np.all(finite >= 0) and np.all(finite <= NOISE.lidar_max_range + 1e-12)


@test("Task 2")
def test_lidar_range_noise_matches_the_setting():
    params = replace(NOISE, n_beams=8, lidar_p_dropout=0.0)
    rng = make_rng(2006)
    clean = lidar_scan(GOAL_POSE, ROOM, params, rng, add_noise=False)[1][0]
    samples = np.array([lidar_scan(GOAL_POSE, ROOM, params, rng)[1][0] for _ in range(1200)])
    assert abs(samples.std() - params.sigma_lidar) < 0.2 * params.sigma_lidar, samples.std()
    assert abs(samples.mean() - clean) < 0.01                # noise is zero mean


@test("Task 2")
def test_lidar_dropout_happens_at_about_the_stated_rate():
    # A high dropout rate makes the count easy to measure without needing huge samples.
    params = replace(NOISE, lidar_p_dropout=0.25, lidar_max_range=20.0)
    _, ranges = lidar_scan(np.array([4.3, 0.6, 0.0]), ROOM, params, make_rng(2007))
    # With a 20 m range nothing is out of reach, so every inf must be a dropout.
    dropped = np.mean(~np.isfinite(ranges))
    assert abs(dropped - 0.25) < 0.1, dropped


# ----------------------------------------------------------------- landmarks
@test("Task 2")
def test_a_noise_free_landmark_reading_is_exact():
    pose = np.array([4.3, 0.6, 0.4])
    for i, r, b in observe_landmarks(pose, ROOM, NOISE.scaled(0.0), make_rng(2008)):
        dx, dy = ROOM.landmarks[i] - pose[:2]
        assert close(r, np.hypot(dx, dy), 1e-12)
        assert close(b, wrap_to_pi(np.arctan2(dy, dx) - pose[2]), 1e-12)


@test("Task 2")
def test_landmark_readings_are_well_formed():
    rng = make_rng(2009)
    for pose in (START_POSE, GOAL_POSE, np.array([4.3, 0.6, 1.2])):
        for i, r, b in observe_landmarks(pose, ROOM, NOISE, rng):
            assert 0 <= i < len(ROOM.landmarks)              # a real landmark id
            assert r > 0
            assert -np.pi - 1e-9 < b <= np.pi + 1e-9         # bearing is wrapped


@test("Task 2")
def test_landmarks_beyond_the_range_limit_are_dropped():
    pose = np.array([4.0, 3.0, 0.0])
    short = replace(NOISE, lm_max_range=1.0, lm_p_detect=1.0, lm_occlusion=False)
    assert observe_landmarks(pose, ROOM, short, make_rng(2010), add_noise=False) == []


@test("Task 2")
def test_the_detection_rate_matches_the_setting():
    pose = np.array([4.3, 0.6, 0.0])
    params = replace(NOISE, lm_p_detect=0.6, sigma_r0=0.0, sigma_r1=0.0, sigma_b=0.0)
    rng = make_rng(2011)
    visible = len(observe_landmarks(pose, ROOM, replace(params, lm_p_detect=1.0), rng))
    assert visible > 0
    trials = 800
    seen = sum(len(observe_landmarks(pose, ROOM, params, rng)) for _ in range(trials))
    rate = seen / (trials * visible)
    assert abs(rate - 0.6) < 0.05, rate


@test("Task 2")
def test_landmark_range_noise_grows_with_distance():
    # sigma = sigma_r0 + sigma_r1 * r, so a landmark twice as far away is noisier.
    rng = make_rng(2012)
    pose = np.array([4.3, 0.6, 0.0])
    params = replace(NOISE, lm_p_detect=1.0, lm_occlusion=False)
    truth = {i: r for i, r, _ in observe_landmarks(pose, ROOM, params.scaled(0.0), rng)}
    near_id = min(truth, key=truth.get)
    far_id = max(truth, key=truth.get)
    assert truth[far_id] > truth[near_id] * 1.5, "need two landmarks at clearly different ranges"

    spread = {}
    for target in (near_id, far_id):
        errs = [o[1] - truth[target] for _ in range(1500)
                for o in observe_landmarks(pose, ROOM, params, rng) if o[0] == target]
        spread[target] = np.std(errs)
        expected = params.sigma_r0 + params.sigma_r1 * truth[target]
        assert abs(spread[target] - expected) < 0.25 * expected, (target, spread[target], expected)
    assert spread[far_id] > spread[near_id]


@test("Task 2")
def test_landmark_bearing_noise_matches_the_setting():
    rng = make_rng(2013)
    pose = np.array([4.3, 0.6, 0.0])
    params = replace(NOISE, lm_p_detect=1.0, lm_occlusion=False)
    truth = {i: b for i, _, b in observe_landmarks(pose, ROOM, params.scaled(0.0), rng)}
    target = min(truth)
    errs = [wrap_to_pi(o[2] - truth[target]) for _ in range(2000)
            for o in observe_landmarks(pose, ROOM, params, rng) if o[0] == target]
    assert abs(np.std(errs) - params.sigma_b) < 0.2 * params.sigma_b, np.std(errs)


# ---------------------------------------------------------------- occlusion
@test("Task 2")
def test_the_partition_hides_landmarks_behind_it():
    # From the goal, L0 in the far corner of the living room is behind the partition wall.
    pose = GOAL_POSE
    perfect = replace(NOISE, lm_p_detect=1.0)
    seen = {i for i, _, _ in observe_landmarks(pose, ROOM, perfect, make_rng(2014),
                                               add_noise=False)}
    assert 0 not in seen, "L0 should be hidden by the partition wall"

    # With occlusion switched off it comes back, which shows the range limit is not the reason.
    no_occlusion = replace(perfect, lm_occlusion=False)
    seen_open = {i for i, _, _ in observe_landmarks(pose, ROOM, no_occlusion, make_rng(2014),
                                                    add_noise=False)}
    assert 0 in seen_open


@test("Task 2")
def test_low_furniture_blocks_the_lidar_but_not_the_landmark_camera():
    # From the start pose, the tv stand sits between the robot and L1 in the near corner.
    pose = START_POSE
    lm = ROOM.landmarks[1]
    dx, dy = lm - pose[:2]
    bearing = np.arctan2(dy, dx)
    distance = np.hypot(dx, dy)

    # The lidar, which sees everything, stops short at the tv stand.
    assert cast_ray(pose[:2], bearing, ROOM) < distance - 0.05
    # The camera, which only cares about tall obstacles, has a clear line.
    assert cast_ray(pose[:2], bearing, ROOM, full_height_only=True) > distance

    seen = {i for i, _, _ in observe_landmarks(pose, ROOM, replace(NOISE, lm_p_detect=1.0),
                                               make_rng(2015), add_noise=False)}
    assert 1 in seen, "L1 should be visible over the top of the tv stand"


@test("Task 2")
def test_turning_occlusion_off_can_only_reveal_more_landmarks():
    rng_a, rng_b = make_rng(2016), make_rng(2016)
    on = replace(NOISE, lm_p_detect=1.0, lm_occlusion=True)
    off = replace(on, lm_occlusion=False)
    for pose in (START_POSE, GOAL_POSE, np.array([4.3, 0.6, 0.9])):
        seen_on = {i for i, _, _ in observe_landmarks(pose, ROOM, on, rng_a, add_noise=False)}
        seen_off = {i for i, _, _ in observe_landmarks(pose, ROOM, off, rng_b, add_noise=False)}
        assert seen_on <= seen_off, pose


# ----------------------------------------------------------------- odometry
@test("Task 2")
def test_noise_free_odometry_matches_the_clean_model():
    rng = make_rng(2017)
    pose = np.array([1.0, 2.0, 0.3])
    for v, w in [(0.4, 0.3), (0.0, 0.8), (-0.2, 0.0)]:
        assert poses_close(odom_step(pose, v, w, DT, NOISE.scaled(0.0), rng),
                           unicycle_step(pose, v, w, DT))


@test("Task 2")
def test_odometry_noise_matches_the_alpha_formula():
    rng = make_rng(2018)
    for v, w in [(0.45, 0.30), (0.0, 0.70), (0.50, 0.0)]:
        samples = np.array([sample_odometry(v, w, NOISE, rng) for _ in range(6000)])
        for column, expected in [
                (0, np.sqrt(NOISE.a1 * v**2 + NOISE.a2 * w**2)),
                (1, np.sqrt(NOISE.a3 * v**2 + NOISE.a4 * w**2)),
                (2, np.sqrt(NOISE.a5 * v**2 + NOISE.a6 * w**2))]:
            measured = samples[:, column].std()
            assert abs(measured - expected) < 0.2 * expected + 1e-9, (v, w, column, measured, expected)


@test("Task 2")
def test_a_stationary_robot_picks_up_no_odometry_noise():
    # The noise scales with the commanded speeds, so standing still must add nothing at all.
    rng = make_rng(2019)
    for _ in range(50):
        assert close(sample_odometry(0.0, 0.0, NOISE, rng), (0.0, 0.0, 0.0))


@test("Task 2")
def test_doubling_the_noise_scale_doubles_the_spread():
    rng = make_rng(2020)
    v, w = 0.45, 0.30
    single = np.array([sample_odometry(v, w, NOISE.scaled(1.0), rng) for _ in range(6000)])
    double = np.array([sample_odometry(v, w, NOISE.scaled(2.0), rng) for _ in range(6000)])
    ratio = double[:, 0].std() / single[:, 0].std()
    assert abs(ratio - 2.0) < 0.2, ratio


@test("Task 2")
def test_a_noise_scale_of_zero_is_completely_deterministic():
    a = run_drive(CMDS, NOISE.scaled(0.0), seed_stream=1, scan_every=10**6)
    b = run_drive(CMDS, NOISE.scaled(0.0), seed_stream=999, scan_every=10**6)
    # Different seeds, but with no noise the odometry must land on the true pose either way.
    assert close(a["odom"], a["true"], 1e-9)
    assert close(a["odom"], b["odom"], 1e-9)


# ------------------------------------------------------------ reference drive
@test("Task 2")
def test_the_reference_drive_reaches_the_goal():
    assert poses_close(LOG["true"][-1], GOAL_POSE, 1e-9), LOG["true"][-1]


@test("Task 2")
def test_the_reference_path_clears_the_furniture():
    worst = min(ROOM.clearance(p[:2]) for p in LOG["true"])
    assert worst >= ROBOT_RADIUS, f"closest approach was {worst:.3f} m"


@test("Task 2")
def test_the_drive_actually_sees_landmarks_along_the_way():
    # Task 4 has nothing to work with if the robot is blind for most of the run.
    seen = np.array([len(o) for o in LOG["obs"]])
    assert seen.mean() > 1.0, seen.mean()
    assert np.mean(seen == 0) < 0.35, np.mean(seen == 0)


@test("Task 2")
def test_the_same_seed_gives_an_identical_run():
    a = run_drive(CMDS, NOISE, seed_stream=42, scan_every=10**6)
    b = run_drive(CMDS, NOISE, seed_stream=42, scan_every=10**6)
    assert close(a["odom"], b["odom"], 0.0)


@test("Task 2")
def test_different_seeds_give_different_runs():
    a = run_drive(CMDS, NOISE, seed_stream=42, scan_every=10**6)
    b = run_drive(CMDS, NOISE, seed_stream=43, scan_every=10**6)
    assert not close(a["odom"], b["odom"], 1e-6)


@test("Task 2")
def test_dead_reckoning_drifts_away_from_the_truth():
    # The whole point of Task 4 is that this error exists and is worth removing.
    assert final_err > 0.05, final_err


@test("Task 2")
def test_more_noise_means_more_drift():
    medians = [np.median(sweep[s][:, -1]) for s in SCALES]
    assert medians[0] == 0.0                       # scale 0 is exact
    assert medians[-1] > medians[1], medians       # scale 2 is clearly worse than scale 0.5


run_tests("Task 2")

---
## Running every test together

The per task blocks above already ran their own groups. This runs the lot in one go, which is
the check to use after changing anything shared, like the room layout or the noise settings.

In [ ]:
groups = sorted({g for g, _, _ in TESTS})
print(f"{len(TESTS)} tests registered across {len(groups)} groups: {', '.join(groups)}\n")
run_tests()

In [ ]:
print("Tasks 1 and 2 summary")
print("=" * 58)
print(f"room                 {ROOM.width:.0f} x {ROOM.height:.0f} m, "
      f"{len(ROOM.furniture)} obstacles, {len(ROOM.landmarks)} landmarks")
print(f"grid                 {NX} x {NY} cells at {GRID_RES} m")
print(f"reference drive      {path_len:.2f} m in {len(CMDS) * DT:.1f} s, "
      f"min clearance {min_clear:.2f} m")
print(f"dead reckoning error {final_err:.3f} m and {head_err:.1f} deg at the goal")
print(f"landmarks per step   mean {seen.mean():.2f}, "
      f"none on {100 * np.mean(seen == 0):.0f}% of steps")
print(f"tests                {len(TESTS)} registered, all passing")

---
## Where Tasks 1 and 2 got to

**Task 1.** A unicycle model integrated exactly as a circular arc, with a differential drive
layer on top providing the wheel and twist conversions and a speed clamp that keeps the path
curvature intact. Checked against four cases worked out by hand (straight line, turn in place,
quarter circle, full circle), all matching to better than $10^{-9}$ over 1000 steps each, and
shown to be independent of step size where forward Euler is not.

**Task 2.** An 8 m by 6 m household room with nine obstacles arranged around a single 1.2 m
doorway, five landmarks, and both arm targets marked. Odometry noise goes in at the velocity
level; landmark and lidar noise go straight onto the readings. All of it is controlled by one
`NoiseParams` object with a single scale multiplier. The measured spreads match what we asked
for, and dead reckoning drifts by a few tens of centimetres over the reference drive, which is
the error Task 4 is meant to remove.

### Simplifications we made on purpose

These are all things to state plainly in the writeup rather than hope nobody asks about.

- **Landmark correspondence is known.** Each reading arrives with the id of the landmark it
  came from, so Task 4 does no data association. Real beacons would need to be told apart
  first, and getting that wrong is a common way for a filter to diverge.
- **Obstacles are axis aligned rectangles.** This makes ray casting exact and fast, but it
  rules out round or angled furniture, and it makes the room tidier than a real one.
- **The noise is zero mean and Gaussian.** A real base also has systematic bias, for instance
  from slightly unequal wheel radii or a sloping floor, and no amount of filtering removes
  that. We do not model it.
- **Sensing is 2D at a single height.** The lidar cannot see a tabletop it would happily drive
  underneath. The two sensor heights we do model are a coarse stand in for this.
- **The reference path is hand made.** The waypoints in section 2.5 were chosen by eye. Task 3
  replaces them with a planned path, and the assertion that the path clears the furniture is
  what will keep the planner honest.